In [ ]:
# Import Basic Packgaes 
import numpy as np
import pandas as pd
from datetime import datetime

import statsmodels as sm
import itertools
import glob
import os
from scipy.signal import argrelextrema
from scipy.signal import find_peaks, peak_widths
from scipy import interpolate

# Data Visualization
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns # advanced vizs
from pandas.plotting import lag_plot


# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

# Set Color Palettes
palette1 = itertools.cycle(sns.color_palette(palette='Set1'))
palette2 = itertools.cycle(sns.color_palette(palette='Set2'))


In [ ]:
directory_name = "D:\\1.5inch Pump Data\\With Vibration Sensor Test\\01.T04_S1_P1_15_100-20\\P1_S1_CSV Files"

In [ ]:
os.chdir(directory_name)

In [ ]:
os.getcwd()

In [ ]:
# current directory csv files
csvs = [x for x in os.listdir(directory_name) if x.endswith('.csv')]
# stats.csv -> stats
fns = [os.path.splitext(os.path.basename(x))[0] for x in csvs]

dic_csv = {}
for i in range(len(fns)):
    dic_csv[fns[i]] = pd.read_csv(csvs[i])

In [ ]:
dic_csv.keys()

In [ ]:
dic_csv['01_P1_1.5_100-20_17May24']

In [ ]:
for key, df in dic_csv.items() :
    df.drop(columns=['P3-Water Suction Flowrate','P3-Water Discharge Flowrate','P3-Air Supply Flowrate','P3-Channel-3'], axis=1, inplace=True)

In [ ]:
for key, df in dic_csv.items():
    df['Cycle-detect'] *=180

In [ ]:
#Set time index
#for key, df in dic_csv.items():
    #df.reset_index(inplace=True)

In [ ]:
for key, df in dic_csv.items():
    df['Time'] =pd.to_datetime(df['Time'])

In [ ]:
for key, df in dic_csv.items():
    df.set_index(['Time'], inplace=True)

In [ ]:
# Plot amplitude vs. time
plt.figure(figsize=(30,7))
#plt.xticks((np.arange(0,370,0.0336669699727025)).astype(int))
plt.plot(dic_csv['04_T02_D1_P2_15_100-20_16Mar24'].index,dic_csv['04_T02_D1_P2_15_100-20_16Mar24']['P2-Air Supply Pressure'],marker = '.')
plt.plot(dic_csv['04_T02_D1_P2_15_100-20_16Mar24'].index,dic_csv['04_T02_D1_P2_15_100-20_16Mar24']['P2-CPM'])
plt.xlabel('Hrs')
plt.ylabel('ASP & CPM')
plt.title('P2_D1_15_16Mar24_T3')

In [ ]:
#Create new dictionary to store the sliced data frame
slice_dic = {}

In [ ]:
# Data frame slice range
start_time = pd.to_datetime('12:30:00.000')
end_time = pd.to_datetime('13:59:59.999')
time_interval = pd.Timedelta(minutes=1)

In [ ]:
# plot sliced df
current_time = start_time
while current_time <= end_time:
    next_time = current_time + time_interval
        
    for key, df in dic_csv.items():
        # Slice the DataFrame based on the specified time range
        sliced_df = df.between_time(current_time.time(), next_time.time())
        # Store the sliced DataFrame in the new dictionary
        slice_dic[key] = sliced_df

        a = len(slice_dic)  # number of rows
        b = 1  # number of columns
        c = 1  # initialize plot counter
        fig1 = plt.figure(figsize=(90,20))

        SMALL_SIZE = 40
            
        for key, df in slice_dic.items():
            # Create a subplot and save the plot
            plt.subplot(a, b, c)
            #plt.vlines(x=df.index, ymin=0, ymax=df["P2-Cycle-detect"], linestyles='dashdot',colors='blue')
            #plt.xticks(np.arange(current_time,next_time,20))
            plt.plot(df.index, df['P2-CPM'],color='black')
            #plt.plot(df.index, df['P3-Water Suction Pressure'])
            #plt.plot(df.index, df['P3-Water Discharge Pressure'],color='magenta')
            plt.plot(df.index, df['P2-Air Supply Pressure'],color='blue')
            #plt.plot(df.index, df['P3-Water Discharge Flowrate'],color='maroon')
            #plt.plot(df.index, df['P3-Air Supply Flowrate'],color='lime')             
            #plt.ylim(65, 110)
            #plt.title("IP-100psi_BP-50psi")
            #plt.legend(df.columns, loc='lower center', bbox_to_anchor=(0.5, -0.14), ncol=8)
            c = c + 1

            plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
            plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
            plt.rc('axes', labelsize=SMALL_SIZE)    # fontsize of the x and y labels
            plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
            plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
                # Save the plot with a filename indicating the time interval

        filename = os.path.join(directory_name, f'{key}_{current_time.strftime("%H-%M-%S")}_{next_time.strftime("%H-%M-%S")}.png')
        plt.savefig(filename)
        plt.close()
            
        current_time = next_time

In [ ]:
# Define FFT parameters
dt = 0.03
f1, f2 = 1000, 1000  # Hz
diff = 100
SMALL_SIZE = 14

In [ ]:
current_time = start_time
while current_time <= end_time:
    next_time = current_time + time_interval
    
    for key, df in dic_csv.items():
        # Slice the DataFrame based on the specified time range
        sliced_df = df.between_time(current_time.time(), next_time.time())
        # Store the sliced DataFrame in the new dictionary
        slice_dic[key] = sliced_df 

        # Perform FFT analysis only if there is data in the sliced DataFrame
        n = len(sliced_df)
        if n > 0:
            t = sliced_df.index
            s = sliced_df['Left-Vibration-Data']

            # FFT
            s -= s.mean()
            fr = np.fft.rfftfreq(n, dt)
            fou = np.fft.rfft(s)

            # Plot and save the FFT plot
            plt.figure(figsize=(28, 10))
                
            plt.subplot(211)
            plt.plot(t, s)
            plt.title('Data with Noise')

            plt.subplot(212)
            plt.plot(np.fft.fftshift(fr), np.fft.fftshift(np.abs(fou)))
            plt.xticks(np.arange(min(np.fft.fftshift(fr)), max(np.fft.fftshift(fr)), 0.4))
            #plt.ylim(0, 6000)
            plt.title('Spectrum of Noisy Data')
            plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
            plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
            plt.rc('axes', labelsize=SMALL_SIZE)    # fontsize of the x and y labels
            plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
            plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels

            # Save the plot with a meaningful filename
            filename = os.path.join(directory_name, f'{current_time.strftime("%H-%M-%S")}_{next_time.strftime("%H-%M-%S")}_FFT.png')
            plt.savefig(filename)
            plt.close()

        else:
            # Handle the case where the DataFrame is empty (e.g., print a message)
            print(f"No data in the time range {current_time} - {next_time}")
        
    current_time = next_time